# SENTINEL / AERIS_64 — Train on Google Colab

**Runtime → Change runtime type → GPU (T4 / L4)**

Pipeline: `prepare_data.py` → `python -m sentinel.cli train` → checkpoints under `runs/`


## 1) GPU check + clone

In [ ]:
!nvidia-smi -L || true
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

import os
os.chdir('/content')
!rm -rf AERIS_64
!git clone --depth 1 https://github.com/giridhari-code/AERIS_64.git
%cd AERIS_64
!pwd && ls configs scripts/data scripts/train packages/sentinel | head


## 2) Install package

In [ ]:
# Colab already has torch; install the rest
%pip install -q safetensors pyyaml tqdm tokenizers numpy
%pip install -q -e ".[data]"
import os
os.environ['PYTHONPATH'] = '/content/AERIS_64/packages'
print('PYTHONPATH=', os.environ['PYTHONPATH'])


## 3) Data

Default: repo `data/samples/*.txt`.  
Apna data: left panel se upload karke `data/raw/` mein rakho, phir cell mein path change karo.


In [ ]:
from pathlib import Path
Path('data/raw').mkdir(parents=True, exist_ok=True)

# Prefer user uploads under data/raw; else samples
raw = list(Path('data/raw').glob('*.txt'))
samples = list(Path('data/samples').glob('*.txt'))
inputs = raw if raw else samples
print('files:', len(inputs))
for p in inputs[:10]:
    print(' -', p, p.stat().st_size, 'bytes')

assert inputs, 'No .txt found. Upload files to data/raw/'

inp = ' '.join(f'"{p}"' for p in inputs)
!python scripts/data/prepare_data.py --input {inp} --out-dir data/tokens --vocab-size 8000 --val-frac 0.05
!ls -lh data/tokens/


## 4) Colab-friendly config (chhota model, T4-safe)

Free Colab pe `d_model=256` + long seq OOM ho sakta hai — isliye chhota preset.


In [ ]:
from pathlib import Path
Path('configs/colab.yaml').write_text('''
model:
  d_model: 128
  d_k: 32
  d_v: 32
  n_skills: 4
  top_k: 2
  k_max: 3
  dendrite_window: 3
  vocab_size: 8000
  max_seq_len: 1024
  tie_embeddings: true
  dropout: 0.05

training:
  batch_size: 4
  grad_accum_steps: 2
  seq_len: 128
  lr: 5.0e-4
  weight_decay: 0.01
  warmup_steps: 50
  max_steps: 2000
  grad_clip: 1.0
  lambda_err: 0.1
  lambda_err_warmup_steps: 200
  lambda_bal: 0.01
  replay_every: 50
  replay_eta: 0.02
  truncate_write_window: 16
  memory_lr_scale: 0.5
  ema_decay: 0.999
  device: cuda
  seed: 42
  log_every: 20
  eval_every: 200
  checkpoint_every: 500
  output_dir: runs/colab
  precision: bf16

data:
  train_path: data/tokens/train.bin
  val_path: data/tokens/val.bin
  tokenizer: hf

safety:
  enable_audit: true
  max_write_norm: 10.0
  max_memory_norm: 50.0
  gate_cap: 1.0
  gate_floor: 0.05
  gate_target: 0.35
  lambda_gate: 0.01
  lambda_ent: 0.005
  anomaly_z_threshold: 4.0
  freeze_on_anomaly: false
  per_session_state: true
  truncate_write_window: 16

logging:
  level: INFO
  wandb: false
'''.strip() + '\n')
print('wrote configs/colab.yaml')


## 5) Train

Pehli baar smoke test: `--max-steps 100`.  
Full Colab run: 2000 steps (config default).


In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/AERIS_64/packages'

# Smoke (fast). Full train ke liye --max-steps hata do ya 2000 rakho.
!python -m sentinel.cli train \
  --config configs/colab.yaml \
  --data-dir data/tokens \
  --output-dir runs/colab \
  --device cuda \
  --precision bf16 \
  --batch-size 4 \
  --seq-len 128 \
  --max-steps 500 \
  --num-workers 0


## 6) Resume (disconnect ke baad)

In [ ]:
# Runtime disconnect ho to same output-dir se continue:
# !python -m sentinel.cli train --config configs/colab.yaml --data-dir data/tokens \
#   --output-dir runs/colab --resume --max-steps 2000 --device cuda --precision bf16 --num-workers 0
print('Uncomment above to resume')


## 7) Download checkpoints

In [ ]:
!ls -lh runs/colab/ 2>/dev/null || ls -lh runs/
# Zip final export for download
!cd runs/colab && zip -r /content/sentinel_colab_ckpt.zip final best checkpoint_latest.pt config.yaml 2>/dev/null || true
from google.colab import files
import os
path = '/content/sentinel_colab_ckpt.zip'
if os.path.isfile(path):
    files.download(path)
else:
    print('No zip yet — check runs/colab after training finishes')


## Tips

| Problem | Fix |
|---------|-----|
| CUDA OOM | `--batch-size 2 --seq-len 64` |
| bf16 error (old GPU) | `--precision fp32` |
| Slow tokenize | chhota `--vocab-size 4000` |
| Apna data | Files → `data/raw/*.txt` upload, cell 3 dubara chalao |

Official path docs: `scripts/train/README.md` on [main](https://github.com/giridhari-code/AERIS_64).
